## MLlib Regression Experiment with Caching

### University of Virginia
### DS 7200: Distributed Computing
### Last Updated: September 22, 2026

---  


#### INSTRUCTIONS: 

This notebook demonstrates construction of a data pipeline and fitting a linear regression model. It illustrates the runtime difference with and without training data caching.

Carefully review the code below, fill in the missing sections, run the code, and note the results

---

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline
import time

# =========================================================
# 0 | SET CONFIGS

LABEL_COLUMN = "median_house_value"
FILENAME_DATA = "california_housing_spark_10000.csv"
MAX_ITER = 20

# =========================================================
# 1 | START SPARK

spark = (
    SparkSession.builder
    .appName("CachingDemo")
    .getOrCreate()
)


# =========================================================
# 2 | READ DATA

# local run: reading the CSV I scp'd from Rivanna
# (/standard/ds7200-apt4c/large_datasets/...) rather than that HPC path
# header=True reads column names from row 1; inferSchema=True lets
# Spark infer types instead of everything defaulting to string

df = spark.read.csv(FILENAME_DATA, header=True, inferSchema=True)

print("Number of records:", df.count())

df.printSchema()


# =========================================================
# 3 | FEATURES

features = [
    "median_income",
    "housing_median_age",
    "total_rooms",
    "total_bedrooms",
    "population",
    "households",
    "latitude",
    "longitude"
]


# =========================================================
# 4 | FEATURE ENGINEERING

# The vector assembler object should be named: assembler
# VectorAssembler: Spark ML models don't take separate feature columns
# like sklearn/R do -- they expect ONE column of vector type. This packs
# the 8 named columns into a single vector column. No learning happens
# here, it's a pure reshape. Named the output "raw_features" (not
# "features") on purpose -- see note below.

assembler = VectorAssembler(inputCols=features, outputCol="raw_features") # inputCols is the list of source columns; outputCol is the name of the new vector column it creates

# The scaler object should be named: scaler
# StandardScaler: scales to unit variance (withStd=True) but does NOT
# center on the mean (withMean=False) -- centering skipped intentionally
# for this exercise. inputCol="raw_features" reads the assembler's
# output; outputCol="features" writes the scaled result.

# KEY DESIGN NOTE: Spark transformers can never overwrite their input
# column in place (dataframes are immutable) -- outputCol must always
# be a new name. By naming assembler's output "raw_features" instead
# of "features", the name "features" is freed up for the scaler's
# output. This means step 5's .select("features", ...) and step 7's
# featuresCol="features" both correctly grab the SCALED vector --
# if assembler had used outputCol="features" directly, the scaled
# output would've needed a different name (e.g. "scaledFeatures") and
# everything downstream would need to reference that name instead, or
# risk silently training on unscaled data.
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=False) 
# inputCol="raw_features": this operates on the vector column that the assembler just made, not the original 8;
# withStd=True: divide by standard deviation (this is the scaling)
# withMean=False: don't subtract mean first. Normally standardization is (x-mean)/std but we are skipping the centering step and just doing x/std

# A pipeline is used to preprocess the data. This eliminates multiple calls to fit(), simplifies the process,
# and it's reusable.
# Pipeline = tidymodels' workflow() equivalent: bundles preprocessing
# steps into one fit-able/reusable object. Runs stages in order:
# assembler first (raw_features), then scaler on its output (features).

pipeline = Pipeline(
    stages=[
        assembler,
        scaler
    ]
)


# =========================================================
# 5 | CREATE TRAINING DATA

pipeline_model = pipeline.fit(df)

processed = (
    pipeline_model
    .transform(df)
    .select(
        "features",
        "median_house_value"
    )
)

# =========================================================
# 6 | TRAIN / TEST SPLIT

#[INSERT CODE TO RANDOMLY SPLIT THE DATA INTO 80% train / 20% test]

train, test = processed.randomSplit([0.8, 0.2], seed=42)
# randomSplit takes a list of weights (not necessarily just two; could do [.7, .15, .15] for a train/val/test in one call and it hands back that many dfs)
# seed=42 for reproducibility, same idea as set.seed() / random_state 

print("Training records:", train.count())
print("Test records:", test.count())


# =========================================================
# 7 | LINEAR REGRESSION

#[INSERT CODE TO INSTANTIATE A LINEAR REGRESSION MODEL.]
# It should use the features column as predictors, LABEL_COLUMN for the target variable
# No formula interface like R (y ~ .) -- have to point explicitly at
# the vector column (featuresCol) and target column (labelCol).
# regParam defaults to 0 (no regularization) -- triggers a Spark
# warning ("might cause numerical instability and overfitting") but
# is fine here since the point of this notebook is caching behavior,
# not model quality.

lr = LinearRegression(featuresCol="features", labelCol=LABEL_COLUMN, maxIter=MAX_ITER)
# featuresCol="features" points at single vector column the pipeline built
# labeCol=LABEL_COLUMN this is the target, "median_house_value"
# maxIter=MAX_ITER this is a cap on optimization iterations (Spark's LinearRegression uses iterative solvers rather than a closed form normal equation by default)

26/09/23 16:35:49 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Number of records: 10000
root
 |-- median_house_value: double (nullable = true)
 |-- median_income: double (nullable = true)
 |-- housing_median_age: double (nullable = true)
 |-- total_rooms: double (nullable = true)
 |-- total_bedrooms: double (nullable = true)
 |-- population: double (nullable = true)
 |-- households: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)

Training records: 8079
Test records: 1921


In [ ]:
# =========================================================
# 8 | WITHOUT CACHE

# Spark is LAZY: train isn't materialized data, it's a recipe (read CSV
# -> assemble -> scale -> split). Every action (.collect(), .count())
# that touches train forces Spark to replay that WHOLE recipe from
# scratch -- nothing is remembered between loop iterations by default.
#
# Section 8 (no cache): 5 iterations, each one re-reads the CSV,
# re-runs assembler/scaler/split, AND re-fits the model.
#   -> 1.89s total
#
# Section 9 (cached): train.cache() marks train to be kept in memory
# after its first materialization -- but cache() alone is still lazy,
# it does nothing until an action runs. train_cached.count() BEFORE
# starting the timer is what actually forces + stores that first
# computation. After that, the loop only re-does fit + predict each
# iteration -- the expensive read/assemble/scale/split happens once.
#   -> 0.89s total, 2.11x speedup
#
# Fair comparison:
#   no cache: 5 x (read + assemble + scale + split + fit + predict)
#   cached:   1 x (read + assemble + scale + split) + 5 x (fit + predict)
#
# RMSE identical (57768.96) both ways, as expected -- caching only
# changes HOW the data gets computed, not what the data/model is.
#
# unpersist() at the end frees the cached data from memory/disk --
# cleanup, same idea as rm()-ing a large object in R when done with 

print("\n========================================")
print("WITHOUT CACHE")
print("========================================")

start = time.time()

for i in range(5):

    model = lr.fit(train)

    # Force Spark to perform another action (inference) using the training data.
    predictions = model.transform(train)

    rmse = (
        predictions
        .selectExpr(
            "sqrt(avg(pow(median_house_value - prediction, 2))) as rmse"
        )
        .collect()[0]["rmse"]
    )

    print(
        f"Iteration {i + 1}: "
        f"training RMSE = {rmse:.2f}"
    )

no_cache_time = time.time() - start

print("\nTime without cache:", no_cache_time)


WITHOUT CACHE


26/09/23 16:35:50 WARN Instrumentation: [1881c7a5] regParam is zero, which might cause numerical instability and overfitting.


Iteration 1: training RMSE = 57768.96


26/09/23 16:35:50 WARN Instrumentation: [a6145143] regParam is zero, which might cause numerical instability and overfitting.


Iteration 2: training RMSE = 57768.96


...

Iteration 3: training RMSE = 57768.96


26/09/23 16:35:51 WARN Instrumentation: [2318e70a] regParam is zero, which might cause numerical instability and overfitting.


Iteration 4: training RMSE = 57768.96


26/09/23 16:35:51 WARN Instrumentation: [37358488] regParam is zero, which might cause numerical instability and overfitting.


Iteration 5: training RMSE = 57768.96

Time without cache: 1.8866653442382812


In [ ]:

# =========================================================
# 9 | WITH CACHE



print("\n========================================")
print("WITH CACHE")
print("========================================")

# =========================================================
# NOTE TO SELF: caching vs. parallelization vs. distributed computing
# =========================================================
# Parallelization = how much work happens AT ONCE.
# Caching = how many TIMES the same work happens, period.
# They compose: cached data is still processed in parallel whenever
# it IS touched -- caching just cuts how often "touched" happens.
#
# Distributed (Spark) vs. parallel-on-one-machine (doParallel etc.):
# doParallel splits across cores on ONE machine, bounded by that
# machine's RAM/CPU, and typically evaluates eagerly (no separate
# "cache" concept needed). Spark splits across MULTIPLE machines with
# no shared memory -- data too big for one box's RAM can still be
# processed, lost partitions can be recomputed from lineage (fault
# tolerance), and operations like groupBy/joins involve network
# shuffles that single-machine parallelism has no equivalent of.

train_cached = train.cache()

# IMPORTANT:
# Materialize the cache before starting the timer. This is done by prompting an action.
train_cached.count()

start = time.time()

for i in range(5):

    # fit the model using the cached training data
    model = lr.fit(train_cached)

    predictions = model.transform(train_cached)

    rmse = (
        predictions
        .selectExpr(
            "sqrt(avg(pow(median_house_value - prediction, 2))) as rmse"
        )
        .collect()[0]["rmse"]
    )

    print(
        f"Iteration {i + 1}: "
        f"training RMSE = {rmse:.2f}"
    )

cache_time = time.time() - start

print("\nTime with cache:", cache_time)


# =========================================================
# 10 | COMPARISON

print("\n========================================")
print("RESULTS")
print("========================================")

print(f"Without cache: {no_cache_time:.2f} seconds")
print(f"With cache:    {cache_time:.2f} seconds")

if cache_time > 0:
    speedup = no_cache_time / cache_time
    print(f"Speedup:       {speedup:.2f}x")


# =========================================================
# CLEAN UP
# =========================================================

train_cached.unpersist()

spark.stop()


WITH CACHE


26/09/23 16:35:52 WARN Instrumentation: [0b0a3502] regParam is zero, which might cause numerical instability and overfitting.


Iteration 1: training RMSE = 57768.96
Iteration 2: training RMSE = 57768.96


26/09/23 16:35:52 WARN Instrumentation: [4d3c1b99] regParam is zero, which might cause numerical instability and overfitting.
26/09/23 16:35:52 WARN Instrumentation: [4fc12c9c] regParam is zero, which might cause numerical instability and overfitting.


Iteration 3: training RMSE = 57768.96
Iteration 4: training RMSE = 57768.96


26/09/23 16:35:52 WARN Instrumentation: [26215d03] regParam is zero, which might cause numerical instability and overfitting.
26/09/23 16:35:52 WARN Instrumentation: [a39c7fe4] regParam is zero, which might cause numerical instability and overfitting.


Iteration 5: training RMSE = 57768.96

Time with cache: 0.8925578594207764

RESULTS
Without cache: 1.89 seconds
With cache:    0.89 seconds
Speedup:       2.11x


*only cache a dataset if you're going to use it again*